In [25]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import statsmodels.formula.api as smf

In [26]:
def load_and_prepare_run(
    run_id,
    result_dir=""
):
    """
    Load loss, label, FOIF and TracIn results for one experimental run.

    Expected filenames:
        IF_sp1_de01_seed_<run_id>.csv
        TC_sp1_de01_seed_<run_id>.csv
        loss_sp1_de01_seed_<run_id>.csv
        label_sp1_de01_seed_<run_id>.csv
    """

    foif_df = pd.read_csv(
        f"IF_7_3_run{run_id}.csv"
    )
    tracin_df = pd.read_csv(
        f"TC_7_3_run{run_id}.csv"
    )
    loss_df = pd.read_csv(
        f"loss_7_3_run{run_id}.csv"
    )
    label_df = pd.read_csv(
        f"Class_label_7_3_run{run_id}.csv"
    )

    label_df = label_df[
        ["id", "label"]
    ].rename(columns={"id": "Train_ID"})

    foif_df = foif_df.rename(
        columns={"Score": "FOIF_Score"}
    )

    tracin_df = tracin_df.rename(
        columns={"Score": "TracIn_Score"}
    )

    density_df = (
        label_df
        .merge(loss_df, on="Train_ID", validate="one_to_one")
        .merge(foif_df, on="Train_ID", validate="one_to_one")
        .merge(tracin_df, on="Train_ID", validate="one_to_one")
    )


    density_df["Minority"] = density_df["label"].astype(int)

    density_df["Log_Loss"] = np.log1p(
        density_df["Training_Loss"]
    )

    density_df["Log_Abs_TC"] = np.log1p(
        np.abs(density_df["TracIn_Score"])
    )

    density_df["Run"] = run_id

    return density_df

In [27]:
def fit_regression_for_run(density_df, run_id):
    """
    Fit signed-score and magnitude regression models for one run.
    """

    signed_model = smf.ols(
        "TracIn_Score ~ Log_Loss * Minority",
        data=density_df
    ).fit()

    magnitude_model = smf.ols(
        "Log_Abs_TC ~ Log_Loss * Minority",
        data=density_df
    ).fit()

    signed_result = {
        "Run": run_id,
        "Model": "Signed TC",
        "Beta_0_Intercept": signed_model.params["Intercept"],
        "Beta_1_Log_Loss": signed_model.params["Log_Loss"],
        "Beta_2_Minority": signed_model.params["Minority"],
        "Beta_3_Interaction": signed_model.params["Log_Loss:Minority"],
        "Minority_Slope": (
            signed_model.params["Log_Loss"]
            + signed_model.params["Log_Loss:Minority"]
        ),
    }

    magnitude_result = {
        "Run": run_id,
        "Model": "Absolute TC magnitude",
        "Beta_0_Intercept": magnitude_model.params["Intercept"],
        "Beta_1_Log_Loss": magnitude_model.params["Log_Loss"],
        "Beta_2_Minority": magnitude_model.params["Minority"],
        "Beta_3_Interaction": magnitude_model.params[
            "Log_Loss:Minority"
        ],
        "Minority_Slope": (
            magnitude_model.params["Log_Loss"]
            + magnitude_model.params["Log_Loss:Minority"]
        ),
    }

    return signed_result, magnitude_result

In [28]:
seeds = [1,2,3,4,5]

all_regression_results = []
all_run_data = []
all_median_loss_results = []

for seed in seeds:
    print(f"Processing run {seed}...")

    density_df = load_and_prepare_run(
        run_id=seed,
        result_dir=""
    )

    # median_loss_run = (
    # density_df
    # .groupby("Density_Group")["Training_Loss"]
    # .median()
    # )

    # all_median_loss_results.append({
    #     "Run": seed,
    #     "Dense_Median_Loss": median_loss_run.get("Dense", np.nan),
    #     "Sparse_Median_Loss": median_loss_run.get("Sparse", np.nan)
    # })

    signed_result, magnitude_result = fit_regression_for_run(
        density_df=density_df,
        run_id=seed
    )

    all_regression_results.extend([
        signed_result,
        magnitude_result
    ])

    all_run_data.append(density_df)

regression_results_df = pd.DataFrame(
    all_regression_results
)

all_density_df = pd.concat(
    all_run_data,
    ignore_index=True
)

display(regression_results_df)

Processing run 1...
Processing run 2...
Processing run 3...
Processing run 4...
Processing run 5...


,Run,Model,Beta_0_Intercept,Beta_1_Log_Loss,Beta_2_Minority,Beta_3_Interaction,Minority_Slope
0,1,Signed TC,1.854694,0.019703,-1.867197,-0.031860,-0.012157
1,1,Absolute TC magnitude,1.048772,0.006755,-1.036487,0.005123,0.011878
2,2,Signed TC,1.826129,0.049315,-1.837340,-0.060624,-0.011309
3,2,Absolute TC magnitude,1.038751,0.017247,-1.027688,-0.006236,0.011011
4,3,Signed TC,1.862723,-0.099928,-1.872910,0.084457,-0.015471
5,3,Absolute TC magnitude,1.051585,-0.035187,-1.041527,0.050373,0.015186
6,4,Signed TC,1.796700,-0.033005,-1.808913,0.019949,-0.013056
7,4,Absolute TC magnitude,1.028271,-0.011936,-1.016211,0.024722,0.012786
8,5,Signed TC,1.938509,-0.121030,-1.953265,0.109440,-0.011590
9,5,Absolute TC magnitude,1.077619,-0.041528,-1.063119,0.052875,0.011347


In [29]:
median_loss_results_df = pd.DataFrame(
    all_median_loss_results
)

display(median_loss_results_df)

""


In [30]:
coefficient_columns = [
    "Beta_0_Intercept",
    "Beta_1_Log_Loss",
    "Beta_2_Minority",
    "Beta_3_Interaction",
    "Minority_Slope",
]

regression_summary = (
    regression_results_df
    .groupby("Model")[coefficient_columns]
    .agg(["mean", "std"])
)

display(regression_summary)

Beta_0_Intercept           Beta_1_Log_Loss            \
                                  mean       std            mean       std   
Model                                                                        
Absolute TC magnitude         1.049000  0.018443       -0.012930  0.025556   
Signed TC                     1.855751  0.053064       -0.036989  0.073659   

                      Beta_2_Minority           Beta_3_Interaction            \
                                 mean       std               mean       std   
Model                                                                          
Absolute TC magnitude       -1.037006  0.017478           0.025371  0.026415   
Signed TC                   -1.867925  0.054144           0.024272  0.072892   

                      Minority_Slope            
                                mean       std  
Model                                           
Absolute TC magnitude       0.012442  0.001674  
Signed TC                  -0.012716  0.001678

In [27]:
# density_df["Loss_Decile"] = pd.qcut(
#     density_df["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# ) + 1

In [29]:
# decile_label_summary = (
#     density_df
#     .groupby(["Loss_Decile", "Density_Group"])
#     .agg(
#         Count=("Train_ID", "size"),
#         Mean_Loss=("Training_Loss", "mean"),

#         Mean_FOIF=("FOIF_Score", "mean"),
#         FOIF_Positive_Fraction=(
#             "FOIF_Score",
#             lambda x: (x > 0).mean()
#         ),

#         Mean_TracIn=("TracIn_Score", "mean"),
#         TracIn_Positive_Fraction=(
#             "TracIn_Score",
#             lambda x: (x > 0).mean()
#         )
#     )
#     .reset_index()
# )

# print(decile_label_summary)

    Loss_Decile Density_Group  Count  Mean_Loss  Mean_FOIF  \
0             1        Sparse    802   0.000060   0.000014   
1             2         Dense    180   0.001065   0.000147   
2             2        Sparse    618   0.000596   0.000124   
3             3         Dense    668   0.001572   0.000224   
4             3        Sparse    132   0.001508   0.000291   
5             4         Dense    705   0.002036   0.000293   
6             4        Sparse     95   0.002046   0.000461   
7             5         Dense    717   0.002481   0.000367   
8             5        Sparse     83   0.002489   0.000485   
9             6         Dense    709   0.003027   0.000474   
10            6        Sparse     91   0.003031   0.000548   
11            7         Dense    663   0.003836   0.000652   
12            7        Sparse    137   0.003963   0.000727   
13            8         Dense    358   0.005499   0.000978   
14            8        Sparse    442   0.007963   0.001388   
15      